# Aq2/Kq2 — Observable Inspection
---
**Workflow:**
1. **Setup** — Imports, config, load grids from parquet
2. **Loop 1** — Generate 7 PNG panels per observable; save manifest to `fig/observables/manifest.csv`
3. **Loop 2** — Fill `obs_template.tikz` → write `fig/observables/<LABEL>.tikz` (always overwritten)
4. **Post** — Write `fig/observables/all_observables.tex`; check `fig/fig_observables.tex` exists

**Compile PDF** (from project root):
```bash
./compile_docs.sh observables
```
Output: `output/documents/observables.pdf`


In [1]:
import os, importlib
import sys; sys.path.insert(0, '.')
import numpy as np
import pandas as pd
import matplotlib
matplotlib.rcParams.update({
    "font.family":          "sans-serif",
    "font.sans-serif":      ["Helvetica Neue", "Arial", "DejaVu Sans"],
    "axes.spines.top":      False,
    "axes.spines.right":    False,
    "axes.linewidth":       0.6,
    "xtick.major.width":    0.6,
    "ytick.major.width":    0.6,
    "xtick.direction":      "out",
    "ytick.direction":      "out",
    "legend.frameon":       False,
    "figure.facecolor":     "white",
})
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize
from pathlib import Path
from scipy import stats
import statsmodels.nonparametric.smoothers_lowess as sm_lowess

from config import *

✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 36


In [2]:
from lib.agrid import Grid
import config; importlib.reload(config); from config import *
import recipe; importlib.reload(recipe)

✓ config.py v4.0 | obs_model: 21 | obs_sweep: 22 | obs: 36


<module 'recipe' from '/Users/toby/proj/aq2/recipe.py'>

In [3]:
df_check = pd.read_parquet('data/IHFC_obs.parquet', columns=obs_model + ['q'])
missing  = [c for c in obs_model if c not in df_check.columns]
all_nan  = [c for c in obs_model if df_check[c].isna().all()]
assert not missing,  f'MISSING from parquet: {missing}'
assert not all_nan,  f'ALL-NaN columns: {all_nan}'
print(f'✓ {len(obs_model)} features verified in parquet | q_clip_max={q_clip_max}')

✓ 21 features verified in parquet | q_clip_max=0.35


In [4]:
def _recipe_for(label):
    for d in recipe.dd:
        if d.get("label") == label:
            return d
    return {}


def _vrange(d, ref_arr):
    """vmin, vmax from recipe or p5/p95 of ref. Always returns finite values."""
    vr = d.get("v_range", None)
    if vr is not None:
        vmin, vmax = float(vr[0]), float(vr[1])
        if np.isfinite(vmin) and np.isfinite(vmax) and vmin < vmax:
            return vmin, vmax
    valid = ref_arr[~np.isnan(ref_arr)]
    if len(valid) == 0:
        return 0.0, 1.0
    vmin = float(np.percentile(valid, 5))
    vmax = float(np.percentile(valid, 95))
    if not (np.isfinite(vmin) and np.isfinite(vmax)):
        return 0.0, 1.0
    if vmin == vmax:
        vmin -= 0.5; vmax += 0.5
    return vmin, vmax


def _loess(x, y):
    order = np.argsort(x)
    xs, ys = x[order], y[order]
    res = sm_lowess.lowess(ys, xs, frac=LOESS_FRAC,
                           return_sorted=True, it=1,
                           delta=0.01*(xs.max()-xs.min()))
    return res[:, 0], res[:, 1]


def make_crossplot(obs, q, vmin, vmax, label, unit, w, h):
    """hexbin + OLS + LOESS crossplot of observable vs heat flow."""
    fig, ax = plt.subplots(figsize=(w, h))
    mask = ~(np.isnan(obs) | np.isnan(q))
    x, y = obs[mask], q[mask]

    if len(x) < 5:
        ax.text(0.5, 0.5, "Insufficient data", ha="center", va="center",
                transform=ax.transAxes, fontsize=8, color="#999999")
        ax.set_axis_off()
        return fig

    hb = ax.hexbin(x, y, gridsize=HEXBIN_GRIDSIZE,
                   cmap="Greys", mincnt=1, bins="log",
                   extent=[vmin, vmax, q_min, q_max])

    slope, intercept, r, *_ = stats.linregress(x, y)
    xline = np.linspace(vmin, vmax, 300)
    ax.plot(xline, intercept + slope * xline,
            color="#c0392b", lw=1.2, zorder=3, label=f"r = {r:.2f}")

    lo = _loess(x, y)
    ax.plot(lo[0], lo[1], color="#2980b9", lw=1.2, zorder=4, label="LOESS")

    ax.set_xlim(vmin, vmax)
    ax.set_ylim(q_min, q_max)
    ax.set_xlabel(f"{label}" + (f" [{unit}]" if unit else ""), fontsize=8)
    ax.set_ylabel("q (mW m⁻²)", fontsize=8)
    ax.tick_params(labelsize=7)
    ax.legend(fontsize=7, loc="upper right", handlelength=1.2)
    fig.tight_layout(pad=0.4)
    return fig

In [5]:
FIG_OBS_DIR.mkdir(parents=True, exist_ok=True)
print(f"PNG output dir : {FIG_OBS_DIR.resolve()}")
print(f"Template       : {TEMPLATE_PATH.resolve()}")
print(f"Manifest       : {MANIFEST_PATH.resolve()}")
print(f"Wrapper tex    : {FIG_WRAPPER.resolve()}")

PNG output dir : /Users/toby/proj/aq2/fig/observables
Template       : /Users/toby/proj/aq2/fig/observables/obs_template.tikz
Manifest       : /Users/toby/proj/aq2/fig/observables/manifest.csv
Wrapper tex    : /Users/toby/proj/aq2/fig/fig_observables.tex


In [6]:
# ── Reference (IHFC) ──────────────────────────────────────────────────────
ref_df = pd.read_parquet(parquet_ref)
ref = Grid(
    lats=ref_df["lat"].values, lons=ref_df["lon"].values,
    name="IHFC", crs=4326, verbose=False,
    log_file=str(log_dir / "read_ref.log"),
)
for col in ref_df.columns:
    if col not in ("lat", "lon"):
        ref.df[col] = ref_df[col].values
print(f"✓ ref: {len(ref.df)} points, {len(ref.df.columns)} columns")

# ── Antarctica ────────────────────────────────────────────────────────────
ant_df = pd.read_parquet(parquet_ant)
ny_ant = ant_df["y"].nunique(); nx_ant = ant_df["x"].nunique()
ant = Grid(
    lats=ant_df["lat"].values, lons=ant_df["lon"].values,
    x=ant_df["x"].values, y=ant_df["y"].values,
    name="Antarctica", crs=3031, verbose=False,
    regular_grid=(ny_ant, nx_ant),
    log_file=str(log_dir / "read_ant.log"),
)
for col in ant_df.columns:
    if col not in ("lat", "lon", "x", "y"):
        ant.df[col] = ant_df[col].values
print(f"✓ ant: {len(ant.df)} points, {len(ant.df.columns)} columns, "
      f"reshape={ant.reshape_tuple}")

# ── Greenland ─────────────────────────────────────────────────────────────
grl_df = pd.read_parquet(parquet_grl)
ny_grl = grl_df["y"].nunique(); nx_grl = grl_df["x"].nunique()
grl = Grid(
    lats=grl_df["lat"].values, lons=grl_df["lon"].values,
    x=grl_df["x"].values, y=grl_df["y"].values,
    name="Greenland", crs=3413, verbose=False,
    regular_grid=(ny_grl, nx_grl),
    log_file=str(log_dir / "read_grl.log"),
)
for col in grl_df.columns:
    if col not in ("lat", "lon", "x", "y"):
        grl.df[col] = grl_df[col].values
print(f"✓ grl: {len(grl.df)} points, {len(grl.df.columns)} columns, "
      f"reshape={grl.reshape_tuple}")

✓ ref: 30848 points, 42 columns
✓ ant: 1779556 points, 39 columns, reshape=(1334, 1334)
✓ grl: 172360 points, 39 columns, reshape=(556, 310)


In [7]:
# ── Loop 1: generate PNG panels and build manifest ────────────────────────
#
# PNG paths stored in the manifest are ROOT-RELATIVE (fig/observables/LABEL_*.png).
# When written into tikz nodes the FIG_OBS_TEX_PREFIX prefix is used to make
# them relative to fig/ (pdflatex working dir):  observables/LABEL_*.png

rows = []
gridlines_kwargs = {
    "draw_labels": False,
    "linewidth":   0.4,
    "color":       "gray",
    "alpha":       0.5,
    "linestyle":   "--",
}

for label in obs_model:
    d     = _recipe_for(label)
    cmap  = d.get("cmap", "viridis")
    unit  = d.get("unit", "")
    desc  = d.get("description", "")
    refs  = d.get("reference", None)

    vmin, vmax = _vrange(d, ref.df[label].values if label in ref.df.columns else np.array([]))

    base  = str(FIG_OBS_DIR / label)   # root-relative base path (no extension)

    # ── Reference scatter map ──────────────────────────────────────────────
    p_ref = base + "_ref" + FIG_EXT
    c_ref = base + "_ref_cmap" + FIG_EXT
    ref.map(data=label, cmap=cmap, vmin=vmin, vmax=vmax,
            coastlines=True, continents=False, cbar=False,
            ext_cbar=True, save_cbar=c_ref, save_fig=p_ref, 
            no_frame=False, transparent=True, show=False,
            gridlines={"step": 30},gridlines_kwargs = gridlines_kwargs,
            cbarsize=(CBAR_W, CBAR_H),
            figsize=(MAP_W_REF,MAP_H_REF),
            scatter_kwargs={"s": 6, "edgecolors": "none"})

    # ── Antarctica map ─────────────────────────────────────────────────────
    p_ant = base + "_ant" + FIG_EXT
    c_ant = base + "_ant_cmap" + FIG_EXT
    ant.map(data=label, cmap=cmap, vmin=vmin, vmax=vmax,
            coastlines=True, continents=False, cbar=False,
            ext_cbar=True, save_cbar=c_ant, save_fig=p_ant,
            no_frame=False, transparent=True, show=False,
            gridlines={"step": 30},gridlines_kwargs = gridlines_kwargs,
            cbarsize=(CBAR_W, CBAR_H),
            figsize=(MAP_W_ANT,MAP_H_ANT),
            scatter_kwargs={"s": 6, "edgecolors": "none"})

    # ── Greenland map ──────────────────────────────────────────────────────
    p_grl = base + "_grl" + FIG_EXT
    c_grl = base + "_grl_cmap" + FIG_EXT
    grl.map(data=label, cmap=cmap, vmin=vmin, vmax=vmax,
            coastlines=True, continents=False, cbar=False,
            ext_cbar=True, save_cbar=c_grl, save_fig=p_grl,
            gridlines={"step": 10},gridlines_kwargs = gridlines_kwargs,
            figsize=(MAP_W_GRL,MAP_H_GRL),cbarsize=(CBAR_W, CBAR_H),
            no_frame=False, transparent=True, show=False)

    # ── Cross-plot ─────────────────────────────────────────────────────────
    p_xp = base + "_xp" + FIG_EXT
    cross_fig = make_crossplot(
        ref.df[label].values, ref.df["q"].values,
        vmin, vmax, label, unit, MAP_W_XP, MAP_H_XP)
    cross_fig.savefig(p_xp, bbox_inches="tight", pad_inches=0.02)
    plt.close(cross_fig)

    rows.append(dict(
        label=label, cmap=cmap, unit=unit, vmin=vmin, vmax=vmax,
        description=desc,
        refs=("; ".join(refs) if isinstance(refs, list) else (refs or "")),
        p_ref=p_ref,     p_ref_cb=c_ref,
        p_ant=p_ant,     p_ant_cb=c_ant,
        p_grl=p_grl,     p_grl_cb=c_grl,
        p_xp=p_xp,
    ))
    print(f"  ok  {label}")

manifest = pd.DataFrame(rows)
manifest.to_csv(MANIFEST_PATH, index=False)
print(f"\nManifest saved: {MANIFEST_PATH}  ({len(manifest)} rows)")

  ok  MOHO
  ok  MOHO_GRAV
  ok  DEM
  ok  LAB
  ok  FREE_AIR
  ok  BOUGUER
  ok  SI
  ok  GEOID
  ok  REVEAL_S80
  ok  REVEAL_S90
  ok  REVEAL_S70
  ok  REVEAL_S100
  ok  REVEAL_VP60VS70
  ok  REVEAL_VP90VS60
  ok  REVEAL_VP50VS80
  ok  LITH_RHO
  ok  CRUST_RHO
  ok  MAG_SEIS_MOHO
  ok  SEDIMENT
  ok  CTD
  ok  EMAG2_LOG

Manifest saved: fig/observables/manifest.csv  (21 rows)


In [8]:
# ── Loop 2: write per-observable tikz files (always overwritten) ──────────
#
# Image paths in tikz nodes must be relative to fig/ (pdflatex working dir).
# The root-relative path  fig/observables/LABEL_ref.png
# becomes the tikz path   observables/LABEL_ref.png

def _root_to_tex(root_path):
    """Convert a root-relative path string to a pdflatex-relative path.
    fig/observables/X.png  →  observables/X.png
    Returns the original string unchanged if it does not start with 'fig/'.
    """
    p = str(root_path)
    prefix = "fig/"
    if p.startswith(prefix):
        return p[len(prefix):]
    return p


def _tex_escape(s):
    for old, new in [
        ("%",  r"\%"), ("&",  r"\&"), ("#",  r"\#"), ("_",  r"\_"),
        ("$",  r"\$"), ("^",  r"\^{}"), ("~", r"\textasciitilde{}"),
    ]:
        s = s.replace(old, new)
    return s


# Placeholder TikZ nodes for missing panels
_PH_MAP  = (
    "\\node[draw=black!15, fill=black!4, minimum width=85mm, "
    "minimum height=90mm, align=center, "
    "font=\\footnotesize\\sffamily\\color{black!40}] {Not available};"
)
_PH_CBAR = (
    "\\node[draw=black!15, fill=black!4, minimum width=85mm, "
    "minimum height=14mm, align=center, "
    "font=\\footnotesize\\sffamily\\color{black!40}] {};"
)


template = TEMPLATE_PATH.read_text()

for _, row in manifest.iterrows():
    label   = row["label"]
    tex_out = FIG_OBS_DIR / f"{label}.tikz"
    try:
        label   = row["label"]
        tex_out = FIG_OBS_DIR / f"{label}.tikz"

        ref_keys = [k.strip() for k in str(row["refs"]).split(";") if k.strip()]
        refs_tex = ", ".join(f"\\cite{{{k}}}" for k in ref_keys) if ref_keys else "---"

        def _img_node(root_path, w_mm, ph_node):
            """Return a TikZ \includegraphics string, or ph_node if MISSING."""
            if str(root_path) == "MISSING":
                return ph_node
            return f"\\includegraphics[width={w_mm}mm]{{{_root_to_tex(root_path)}}}"

        body = template
        body = body.replace("%%LABEL%%",       _tex_escape(label))
        body = body.replace("%%DESCRIPTION%%", _tex_escape(str(row["description"])))
        body = body.replace("%%REFERENCES%%",  refs_tex)

        body = body.replace("%%PATH_REF%%",    _root_to_tex(row["p_ref"]))
        body = body.replace("%%PATH_REF_CB%%", _root_to_tex(row["p_ref_cb"]))
        body = body.replace("%%PATH_XP%%",     _root_to_tex(row["p_xp"]))

        # Antarctica — may be MISSING
        ant_val    = row["p_ant"]    if row["p_ant"]    != "MISSING" else "MISSING"
        ant_cb_val = row["p_ant_cb"] if row["p_ant_cb"] != "MISSING" else "MISSING"
        grl_val    = row["p_grl"]    if row["p_grl"]    != "MISSING" else "MISSING"
        grl_cb_val = row["p_grl_cb"] if row["p_grl_cb"] != "MISSING" else "MISSING"

        body = body.replace(
            "{%%PATH_ANT%%}",
            "{" + (_root_to_tex(ant_val)    if ant_val    != "MISSING" else "MISSING_ANT")    + "}")
        body = body.replace(
            "{%%PATH_ANT_CB%%}",
            "{" + (_root_to_tex(ant_cb_val) if ant_cb_val != "MISSING" else "MISSING_ANT_CB") + "}")
        body = body.replace(
            "{%%PATH_GRL%%}",
            "{" + (_root_to_tex(grl_val)    if grl_val    != "MISSING" else "MISSING_GRL")    + "}")
        body = body.replace(
            "{%%PATH_GRL_CB%%}",
            "{" + (_root_to_tex(grl_cb_val) if grl_cb_val != "MISSING" else "MISSING_GRL_CB") + "}")

        # Replace MISSING_ tokens with inline placeholder nodes
        body = body.replace(
            "\\includegraphics[width=85mm]{MISSING_ANT}",    _PH_MAP)
        body = body.replace(
            "\\includegraphics[width=85mm]{MISSING_ANT_CB}", _PH_CBAR)
        body = body.replace(
            "\\includegraphics[width=85mm]{MISSING_GRL}",    _PH_MAP)
        body = body.replace(
            "\\includegraphics[width=85mm]{MISSING_GRL_CB}", _PH_CBAR)

        tex_out.write_text(body)
        print(f"  wrote: {tex_out}")
    except Exception as e:
        print(f"  ERROR {label}: {e}")

print(f"\n{len(manifest)} tikz files written to {FIG_OBS_DIR}/")

  wrote: fig/observables/MOHO.tikz
  wrote: fig/observables/MOHO_GRAV.tikz
  wrote: fig/observables/DEM.tikz
  wrote: fig/observables/LAB.tikz
  wrote: fig/observables/FREE_AIR.tikz
  wrote: fig/observables/BOUGUER.tikz
  wrote: fig/observables/SI.tikz
  wrote: fig/observables/GEOID.tikz
  wrote: fig/observables/REVEAL_S80.tikz
  wrote: fig/observables/REVEAL_S90.tikz
  wrote: fig/observables/REVEAL_S70.tikz
  wrote: fig/observables/REVEAL_S100.tikz
  wrote: fig/observables/REVEAL_VP60VS70.tikz
  wrote: fig/observables/REVEAL_VP90VS60.tikz
  wrote: fig/observables/REVEAL_VP50VS80.tikz
  wrote: fig/observables/LITH_RHO.tikz
  wrote: fig/observables/CRUST_RHO.tikz
  wrote: fig/observables/MAG_SEIS_MOHO.tikz
  wrote: fig/observables/SEDIMENT.tikz
  wrote: fig/observables/CTD.tikz
  wrote: fig/observables/EMAG2_LOG.tikz

21 tikz files written to fig/observables/


<>:54: SyntaxWarning: invalid escape sequence '\i'
<>:54: SyntaxWarning: invalid escape sequence '\i'
/var/folders/64/9b359m2n41d9ywlsbcs4s7rc0000gn/T/ipykernel_2557/2282708421.py:54: SyntaxWarning: invalid escape sequence '\i'
  """Return a TikZ \includegraphics string, or ph_node if MISSING."""


In [9]:
# ── Write all_observables.tex (always overwritten) ────────────────────────

manifest = pd.read_csv(MANIFEST_PATH)

with open(ALL_OBS_TEX, "w") as fh:
    fh.write("% Auto-generated by 2_OBSERVABLES.ipynb — do not edit\n\n")
    for label in manifest["label"]:
        fh.write(f"\\input{{observables/{label}.tikz}}\n\n"
                 f"\\vspace{{6mm}}\n\n")
print(f"✓ {ALL_OBS_TEX}  ({len(manifest)} entries)")

✓ fig/observables/all_observables.tex  (21 entries)


In [10]:
# ── Check fig/fig_observables.tex exists ──────────────────────────────────
if FIG_WRAPPER.exists():
    print(f"✓ wrapper exists: {FIG_WRAPPER}")
else:
    print(f"✗ wrapper NOT found: {FIG_WRAPPER}")
    print("  → copy fig_observables.tex to fig/ before compiling")

print("\nDone. To compile:")
print("  ./compile_docs.sh observables")

✓ wrapper exists: fig/fig_observables.tex

Done. To compile:
  ./compile_docs.sh observables


In [11]:
! ./compile_docs.sh


──────────────────────────────────────────────
→ Compiling: fig/fig_observables.tex
This is pdfTeX, Version 3.141592653-2.6-1.40.27 (TeX Live 2025) (preloaded format=pdflatex)
 restricted \write18 enabled.
entering extended mode
(./fig_observables.tex
LaTeX2e <2024-11-01> patch level 2
L3 programming layer <2025-01-18>
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/article.cls
Document Class: article 2024/06/29 v1.4n Standard LaTeX document class
(/usr/local/texlive/2025/texmf-dist/tex/latex/base/size11.clo))
(/usr/local/texlive/2025/texmf-dist/tex/latex/geometry/geometry.sty
(/usr/local/texlive/2025/texmf-dist/tex/latex/graphics/keyval.sty)
(/usr/local/texlive/2025/texmf-dist/tex/generic/iftex/ifvtex.sty
(/usr/local/texlive/2025/texmf-dist/tex/generic/iftex/iftex.sty)))
(/usr/local/texlive/2025/texmf-dist/tex/latex/graphics/graphicx.sty
(/usr/local/texlive/2025/texmf-dist/tex/latex/graphics/graphics.sty
(/usr/local/texlive/2025/texmf-dist/tex/latex/graphics/trig.sty)
(/usr/local/t

In [ ]:
def normalise_refs(val):
    """Convert any reference value to a LaTeX \citet{} string or empty string."""
    if val is None:
        return ''
    if isinstance(val, list):
        if len(val) == 0:
            return ''
        return r'\citet{' + ', '.join(val) + '}'
    if isinstance(val, str) and val.strip():
        return r'\citet{' + val.strip() + '}'
    return ''

def dd_to_df(dd, include_labels=None):
    rows = []
    for d in dd:
        label = d.get('label', '')
        
        # Filter on base label before any grid suffix
        if include_labels is not None and label not in include_labels:
            continue
        
        grid = d.get('grid')
        if grid:
            label = f"{label} ({grid})"
        
        rows.append({
            'Label':       label,
            'Description': d.get('description', ''),
            'References':  normalise_refs(d.get('reference')),
        })
    return pd.DataFrame(rows)

df = dd_to_df(dd, include_labels=obs_model)

In [ ]:
df

In [ ]:
def df_to_latex(df):
    lines = []
    lines.append(r'\begin{tabular}{lp{7cm}l}')
    lines.append(r'\toprule')
    lines.append(r'Label & Description & References \\')
    lines.append(r'\midrule')
    for _, row in df.iterrows():
        label = row['Label'].replace('_', r'\_')
        desc  = row['Description'].replace('_', r'\_')
        refs  = row['References']
        lines.append(f"{label} & {desc} & {refs} \\\\")
    lines.append(r'\bottomrule')
    lines.append(r'\end{tabular}')
    return '\n'.join(lines)

print(df_to_latex(df))